In [2]:
# STEP 1: Load external knowledge base and convert into pages
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path="C:\\Anand\\old_laptop_backup\\D drive data\\Anand\\material\\Training\\Gen_AI\\L2_Applied_Gen_AI\\external_knowledgebase_for_rag\\Python_Programming.pdf")
docs = loader.load()

#print(docs[0].page_content[:50])  # Print first 50 characters of the first page])
print('PDF file is loaded and converted into pages')

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


PDF file is loaded and converted into pages


In [3]:
# STEP 2: Split the external knowledge base i.e. create chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300, 
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)
split_docs = text_splitter.split_documents(docs)
print(f'Number of chunks created: {len(split_docs)}')

Number of chunks created: 528


In [4]:
# STEP 3: Create embeddings & vector stores using FAISS/Chroma etc.
from langchain_community.embeddings import BedrockEmbeddings
from langchain_community.vectorstores import FAISS
import os
from dotenv import load_dotenv, find_dotenv
import boto3

env_path = find_dotenv()
if not env_path:
    raise FileNotFoundError(".env not found")

load_dotenv(env_path)
aws_access_key = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')

bedrock_client = boto3.client(
    service_name="bedrock-runtime",
    region_name="us-east-1",
    aws_access_key_id=aws_access_key,
    aws_secret_access_key=aws_secret_access_key
)

embeddings = BedrockEmbeddings(
    model_id="amazon.titan-embed-text-v2:0",
    client=bedrock_client)

#embeddings = OllamaEmbeddings(model="nomic-embed-text")
faiss_vector_store = FAISS.from_documents(split_docs, embeddings)
print(faiss_vector_store)

In [6]:
# STEP 4: Integrate with LLM & build retrieval chain
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_classic.chains import RetrievalQA
from langchain_aws import ChatBedrock
user_prompt_template = """Use the following pieces of context to answer the question at the end.
If you don't know the answer, just say that you don't know, don't try to make up an answer.
-----------------
{context}
Question: {question}
"""
custom_prompt = PromptTemplate(
    template=user_prompt_template, input_variables=["context", "question"]
)
parser = StrOutputParser()
model = ChatBedrock(model="amazon.nova-micro-v1:0", client=bedrock_client)
retrieval_chain = RetrievalQA.from_chain_type(
    llm=model,
    chain_type="stuff", # Map_reduce, refine, map_rerank, stuff
    retriever=faiss_vector_store.as_retriever(search_kwargs={"k": 6}),
    return_source_documents=True,
    chain_type_kwargs={"prompt": custom_prompt}

)

In [ ]:
# STEP 5: Build UI user interface
import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = retrieval_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Chatbot",
    description="Ask questions about Python programming based on the provided pdf."
)

# Launch the interface
iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
C:\Users\ak60492\AppData\Local\Temp\ipykernel_52256\1039528188.py:6: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = retrieval_chain({"query": message})
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. U